In [10]:
%reload_ext autoreload
%autoreload 2


In [11]:
from dipy.data import get_fnames
from dipy.io.gradients import read_bvals_bvecs
from dipy.io.image import load_nifti

In [12]:
data, affine = load_nifti("data/data.nii.gz")
mask, _ = load_nifti("data/nodif_brain_mask.nii.gz")
bvals, bvecs = read_bvals_bvecs("data/bvals", "data/bvecs")

In [13]:
import numpy as np

import jax
import jax.numpy as jnp

from dmri.simulators import Ball3StickNoise, MultiCompartment, Dot, Ball, Ball3StickSharedDiffusivity
from dmri.simulators.local_signal_models.ball import StaticBall
from dmri.simulators.acquisition_scheme import acquisition_scheme, random_hcp_acquisition
from flax import nnx

sim_type = Ball3StickSharedDiffusivity
dot_ball = sim_type.from_theta(np.random.randn(sim_type.theta_dim))

In [14]:

@jax.jit
def simulator(rng):
    rng0, rng1, rng2, rng3, rng4, rng5 = jax.random.split(rng, 6)
    rng0_1, rng0_2 = jax.random.split(rng0)
    acq = random_hcp_acquisition(rng0_2)
    theta = jax.random.normal(rng1, shape=(sim_type.theta_dim,))
    rng_p, rng_m = jax.random.split(rng2)
    p_mask = jax.random.beta(rng_p, a=1, b=1, shape=(1,))
    model_mask = jax.random.bernoulli(rng_m, p=p_mask, shape=(len(sim_type.model_types)))
    # This needs to have only one entry
    model_noise_idx = jax.random.randint(rng4, minval=0, maxval=len(sim_type.noise_types), shape=(1,))
    model_noise_mask = jnp.zeros(len(sim_type.noise_types), dtype=jnp.bool)
    model_noise_mask = model_noise_mask.at[model_noise_idx].set(True)
    model_mask = jnp.concatenate([model_mask, model_noise_mask], axis=-1)
    ball_stick = sim_type.from_theta(theta, model_mask=model_mask)

    x_o = ball_stick.signal(acq, rng=rng5)
    return p_mask, model_mask, theta, x_o, acq


In [15]:
p_mask,masks, theta, x_o, acq = jax.vmap(simulator)(jax.random.split(jax.random.PRNGKey(0), num=200))

In [16]:
from dmri.nn.dmri_reconstruction_model import DMRIInferenceModel, DMRIInferenceModelConfig, DMRIInferenceModelConfigMaskPriorAmortized, DMRIInferenceModelConfigMaskPriorAmortizedPP
from dmri.nn.tokenizer import DMRITokenizerPP

cfg = DMRIInferenceModelConfigMaskPriorAmortized(sim_type, use_attention_mask=False)
model = DMRIInferenceModel(cfg, nnx.Rngs(1))

In [17]:
nnx.display(model)

In [18]:
model(masks, theta, x_o, acq.bvals, acq.bvecs, p_mask)[1] - theta

(0, 1, 2, 3, 4, 5, 6)


Array([[-0.,  0.,  0., ..., -0., -0.,  0.],
       [-0.,  0.,  0., ..., -0., -0.,  0.],
       [-0.,  0.,  0., ..., -0., -0.,  0.],
       ...,
       [-0.,  0.,  0., ..., -0., -0., -0.],
       [-0.,  0.,  0., ..., -0., -0.,  0.],
       [ 0.,  0.,  0., ..., -0., -0.,  0.]], dtype=float32)

In [15]:
tokens_cfg, y, mask_prior = model.embed_inputs(masks, x_o, acq.bvals, acq.bvecs, mask_prior=p_mask)

(0, 1, 2, 3, 4, 5, 6)


In [130]:
y.shape

(200, 105, 64)

In [131]:
model.model_decoder.loss_fn(masks, model.tokenizer, y=y, context=mask_prior, tokens_cfg=tokens_cfg, permute_order=True, rng=jax.random.PRNGKey(2))

Array(4.625, dtype=float32)

In [132]:
model.loss_fn(jax.random.key(0),model_mask=masks, theta=theta, x=x_o,bvals=acq.bvals, bvecs=acq.bvecs, mask_prior=p_mask)

(0, 1, 2, 3, 4, 5)


Array([4.619, 7.654], dtype=float32)

In [133]:
import optax

scheduler = optax.cosine_onecycle_schedule(100*500, 5e-4)
optimizer = optax.chain(optax.adaptive_grad_clip(50.), optax.ema(0.01), optax.adamw(scheduler))

params = nnx.state(model, nnx.Param)
opt_state = optimizer.init(params)


In [134]:

def loss_fn(params, data, rng):
    nnx.update(model, params)
    p_mask, model_mask, thetas,  xs, bvals, bvecs = data
    return model.loss_fn(rng,model_mask=model_mask, theta=thetas, x=xs,bvals=bvals, bvecs=bvecs, mask_prior=p_mask).sum()


@jax.jit
def update(params, opt_state, data, rng):
    loss, grads = jax.value_and_grad(loss_fn)(params, data, rng)
    updates, opt_state = optimizer.update(grads, opt_state, params=params)
    new_params = optax.apply_updates(params, updates)
    return new_params, opt_state, loss

In [135]:
batch_simulator = jax.jit(jax.vmap(simulator))

In [ ]:
key = jax.random.PRNGKey(42)
for i in range(100):
    key, subkey1 = jax.random.split(key, 2)
    p_mask, masks, thetas, x_os, acq = batch_simulator(jax.random.split(subkey1, 2**16))
    l = 0
    for j in range(50):
        key, subkey2 = jax.random.split(key, 2)
        idx = jax.random.randint(subkey2, (128,), 0, 2**16)
        p_mask_batch, masks_batch, thetas_batch, x_os_batch, bvals_batch, bvecs_batch = p_mask[idx],masks[idx], thetas[idx], x_os[idx], acq.bvals[idx], acq.bvecs[idx]
        params, opt_state, loss = update(params, opt_state, (p_mask_batch,masks_batch, thetas_batch, x_os_batch, bvals_batch, bvecs_batch), subkey2)
        l += loss
    print(l/50)

(0, 1, 2, 3, 4, 5)
9.212436
7.4260955
6.940317
6.642066
6.48148
6.3094554
6.2311172
6.155524
6.063134
6.038691
5.9713817
5.87373
5.815197
5.7173495
5.6595483


In [7]:
from probjax.utils.odeint import odeint
import jax.numpy as jnp

odeint(lambda t,x: x, jnp.ones(10), jnp.linspace(0, 1, 100), filter_state=lambda *args: None)


TypeError: Indexer must have integer or boolean type, got indexer with type float32 at position 0, indexer value []